# Figure aggiuntive — U-Net SWE Downscaling

Produce le 4 figure aggiuntive per il paper:
- **Fig. 7** — Scatter SWE predetto vs osservato per bacino (40 anni OOS)
- **Fig. 8** — Mappa spaziale side-by-side SPORTLIS vs U-Net (1 aprile anno scelto)
- **Fig. 9** — Curve di loss training/validation dai log PBS
- **Fig. 10** — Bias (pred − obs) vs SWE osservato

Eseguire su NCAR JupyterHub con kernel `sportlis-unet`.

In [ ]:
# ── Cell 1: Import e path ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import torch
import torch.nn as nn
import torch.nn.functional as F
import re, json
from pathlib import Path
from scipy.ndimage import uniform_filter
from scipy import stats

# ── Auto-detect environment ───────────────────────────────────────────────
_ON_NCAR  = Path('/glade/u/home/cionni').exists()
_ON_SMCE  = (not _ON_NCAR) and Path('/home/jovyan/efs').exists()
if _ON_NCAR:
    NCAR_HOME         = Path('/glade/u/home/cionni')
    NCAR_SCRATCH      = NCAR_HOME / 'derecho_scratch'
    PROJECT_DIR       = NCAR_HOME / 'unet_sportlis'
    SPORTLIS_DATA_DIR = NCAR_SCRATCH / 'sportlis_swe'
    ZARR_DIR          = NCAR_SCRATCH / 'zarr_extended'
    MEMMAP_DIR        = NCAR_SCRATCH / 'memmap_extended'
    OUTPUT_DIR        = PROJECT_DIR  / 'output_extended'
    AUX_DIR           = PROJECT_DIR  / 'auxiliary'
    LOG_DIR           = PROJECT_DIR  / 'logs'
else:
    PROJECT_ROOT      = Path('/Users/irene/PROJECTS.index/Hydrology/projects/sportlis_project')
    SPORTLIS_DATA_DIR = PROJECT_ROOT / 'data/inputs'
    ZARR_DIR          = PROJECT_ROOT / 'prepared_pp_narr_sportlis_extended'
    MEMMAP_DIR        = PROJECT_ROOT / 'memmap_extended'
    OUTPUT_DIR        = Path('sportlis_unet_extended_loyo_outputs')
    AUX_DIR           = PROJECT_ROOT / 'auxiliary'
    LOG_DIR           = OUTPUT_DIR   / 'logs'

STATIC_FILE = AUX_DIR / 'sportlis_static_extended.nc'
CANOPY_FILE = AUX_DIR / 'sportlis_canopy_extended_3km.nc'
CKPT_TPL    = 'best_unet_extended_LOYO_test{year}.pt'
BASIN_CSV   = OUTPUT_DIR / 'basin_ts_results.csv'
FIG_DIR     = OUTPUT_DIR / 'figures_additional'
FIG_DIR.mkdir(exist_ok=True)

BASINS = {
    'Pacific_NW_Cascades':  ('Pacific NW Cascades',   '#1f77b4'),
    'Oregon_Cascades':      ('Oregon Cascades',        '#17becf'),
    'Sierra_Nevada':        ('Sierra Nevada',           '#ff7f0e'),
    'Northern_Rockies':     ('Northern Rockies',        '#2ca02c'),
    'Southern_Rockies':     ('Southern Rockies (CO)',   '#d62728'),
    'Idaho':                ('Idaho (dom. originale)',  '#9467bd'),
}

ALL_YEARS = list(range(1985, 2025))
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Ambiente: {"NCAR" if _ON_NCAR else "SMCE" if _ON_SMCE else "locale"}')
print(f'Device:   {device}')
print(f'STATIC:   {STATIC_FILE}  esiste={STATIC_FILE.exists()}')
print(f'CANOPY:   {CANOPY_FILE}  esiste={CANOPY_FILE.exists()}')
print(f'MEMMAP:   {MEMMAP_DIR}  esiste={MEMMAP_DIR.exists()}')
print(f'BASIN CSV esiste: {BASIN_CSV.exists()}')

In [ ]:
# ── Cell 2: Carica basin_ts_results.csv ───────────────────────────────────
df = pd.read_csv(BASIN_CSV)
df.columns = df.columns.str.strip()
if 'fold_used' not in df.columns:
    df['fold_used'] = df['year']
    df['is_oos']    = True
df['is_oos'] = df['is_oos'].astype(str).str.lower().isin(['true', '1', 'yes'])
df_oos = df[df['is_oos']].copy()
print(f'Totale righe: {len(df)}  |  OOS: {len(df_oos)}')
print(f'Anni OOS: {sorted(df_oos.year.unique())}')
df_oos.head(3)

In [ ]:
# ── Cell 3: FIG. 7 — Scatter pred vs obs per bacino ───────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for ax, (basin_key, (basin_label, color)) in zip(axes, BASINS.items()):
    sub = df_oos[df_oos['basin'] == basin_key].dropna(subset=['pred_mean', 'obs_mean'])
    if sub.empty:
        ax.set_visible(False)
        continue
    x = sub['obs_mean'].values
    y = sub['pred_mean'].values
    sc = ax.scatter(x, y, c=sub['year'], cmap='RdYlGn_r', s=60,
                    edgecolors='k', linewidths=0.4, zorder=3)
    plt.colorbar(sc, ax=ax, label='Anno', shrink=0.85)
    lim_max = max(x.max(), y.max()) * 1.1
    ax.plot([0, lim_max], [0, lim_max], 'k--', lw=1.2, label='1:1', zorder=2)
    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(0, lim_max, 100)
        ax.plot(xfit, slope * xfit + intercept, color=color, lw=2,
                label=f'fit: y={slope:.2f}x+{intercept:.2f}\nr={r:.2f}', zorder=4)
    ax.set_xlabel('SWE osservato (mm)', fontsize=10)
    ax.set_ylabel('SWE predetto (mm)', fontsize=10)
    ax.set_title(basin_label, color=color, fontweight='bold', fontsize=11)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_xlim(0, lim_max); ax.set_ylim(0, lim_max)
    ax.grid(True, alpha=0.3)

fig.suptitle('Fig. 7 — Scatter SWE predetto vs osservato (OOS LOYO-40)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out7 = FIG_DIR / 'fig7_scatter_pred_obs.png'
fig.savefig(out7, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvato: {out7}')

In [ ]:
# ── Cell 4: FIG. 10 — Bias vs SWE osservato ──────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for ax, (basin_key, (basin_label, color)) in zip(axes, BASINS.items()):
    sub = df_oos[df_oos['basin'] == basin_key].dropna(subset=['pred_mean', 'obs_mean'])
    if sub.empty:
        ax.set_visible(False)
        continue
    obs  = sub['obs_mean'].values
    bias = sub['pred_mean'].values - obs
    idx  = np.argsort(obs)
    obs_s, bias_s = obs[idx], bias[idx]
    ax.scatter(obs_s, bias_s, c=sub['year'].values[idx],
               cmap='RdYlGn_r', s=60, edgecolors='k', linewidths=0.4, zorder=3)
    ax.axhline(0, color='k', lw=1.2, ls='--', zorder=2)
    if len(obs_s) >= 5:
        win = max(3, len(obs_s) // 5)
        ax.plot(np.convolve(obs_s,  np.ones(win)/win, mode='valid'),
                np.convolve(bias_s, np.ones(win)/win, mode='valid'),
                color=color, lw=2.5, label=f'media mobile (w={win})')
    ax.set_xlabel('SWE osservato (mm)', fontsize=10)
    ax.set_ylabel('Bias pred−obs (mm)', fontsize=10)
    ax.set_title(basin_label, color=color, fontweight='bold', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Fig. 10 — Bias (pred−obs) vs SWE osservato — anni OOS LOYO-40',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out10 = FIG_DIR / 'fig10_bias_vs_swe.png'
fig.savefig(out10, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvato: {out10}')

In [ ]:
# ── Cell 5: FIG. 9 — Curve di loss (solo ultimo run valido per fold) ──────
FOLD_SAMPLE = [0, 5, 10, 20, 30, 39]
pat_epoch   = re.compile(r'E(\d+)/\d+\s+train=([\d.]+)\s+val=([\d.]+)')

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, fi in zip(axes, FOLD_SAMPLE):
    log = LOG_DIR / f'loyo40_job{fi}.log'
    if not log.exists():
        ax.set_visible(False)
        continue
    text = log.read_text(errors='ignore')
    # Filtra epoche con loss > 0.05 (scarta run bad pre-fix)
    valid = [(int(m.group(1)), float(m.group(2)), float(m.group(3)))
             for m in pat_epoch.finditer(text)
             if float(m.group(2)) > 0.05 and float(m.group(3)) > 0.05]
    # Prende solo l'ultimo run (il contatore epoca riparte da 1)
    runs, current = [], []
    for ep, tr, vl in valid:
        if current and ep <= current[-1][0]:
            runs.append(current); current = []
        current.append((ep, tr, vl))
    if current: runs.append(current)
    last_run = runs[-1] if runs else []
    if not last_run:
        ax.set_title(f'fold {fi} — nessun run valido', color='gray'); continue

    epochs     = [r[0] for r in last_run]
    train_loss = [r[1] for r in last_run]
    val_loss   = [r[2] for r in last_run]
    test_yr    = (re.search(r'test=(\d{4})', text) or type('', (), {'group': lambda s,i: str(1985+fi)})()).group(1)

    ax.plot(epochs, train_loss, 'o-',  color='steelblue', ms=3, lw=1.5, label='Train loss')
    ax.plot(epochs, val_loss,   's--', color='tomato',    ms=3, lw=1.5, label='Val loss')
    best_ep = epochs[int(np.argmin(val_loss))]
    ax.axvline(best_ep, color='k', lw=1, ls=':', alpha=0.7)
    ax.text(best_ep + 0.3, min(train_loss + val_loss) + 0.02,
            f'best ep={best_ep}\nval={min(val_loss):.3f}', fontsize=7)
    ax.set_xlabel('Epoca', fontsize=9)
    ax.set_ylabel('Loss (Huber, log1p)', fontsize=9)
    ax.set_title(f'Fold test={test_yr}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

fig.suptitle('Fig. 9 — Curve di convergenza (loss train/val) — fold selezionati',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out9 = FIG_DIR / 'fig9_loss_curves.png'
fig.savefig(out9, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvato: {out9}')

In [ ]:
# ── Cell 6: Architettura U-Net (copia esatta da train_loyo40.py) ──────────
def pad_to_mult(x, mult=8):
    _, _, H, W = x.shape
    pH = (mult - H % mult) % mult
    pW = (mult - W % mult) % mult
    return F.pad(x, (0, pW, 0, pH)), (pH, pW)

def unpad(x, pads):
    pH, pW = pads
    H, W = x.shape[-2], x.shape[-1]
    return x[..., :H - pH if pH else H, :W - pW if pW else W]

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        g = min(8, out_ch)
        while out_ch % g != 0: g -= 1
        layers = [nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.GroupNorm(g, out_ch), nn.GELU(),
                  nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(g, out_ch), nn.GELU()]
        if dropout > 0: layers.append(nn.Dropout2d(dropout))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels, out_channels=1, base=32, dropout=0.1):
        super().__init__()
        self.e1=DoubleConv(in_channels,base); self.p1=nn.MaxPool2d(2)
        self.e2=DoubleConv(base,base*2);      self.p2=nn.MaxPool2d(2)
        self.e3=DoubleConv(base*2,base*4);    self.p3=nn.MaxPool2d(2)
        self.bn=DoubleConv(base*4,base*8,dropout=dropout)
        self.u1=nn.ConvTranspose2d(base*8,base*4,2,stride=2); self.d1=DoubleConv(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2); self.d2=DoubleConv(base*4,base*2)
        self.u3=nn.ConvTranspose2d(base*2,base,  2,stride=2); self.d3=DoubleConv(base*2,base)
        self.out=nn.Conv2d(base,out_channels,1)
    def forward(self, x):
        xp, pads = pad_to_mult(x)
        e1=self.e1(xp); e2=self.e2(self.p1(e1)); e3=self.e3(self.p2(e2))
        b=self.bn(self.p3(e3))
        d=self.d1(torch.cat([self.u1(b),e3],1))
        d=self.d2(torch.cat([self.u2(d),e2],1))
        d=self.d3(torch.cat([self.u3(d),e1],1))
        return F.softplus(unpad(self.out(d),pads))

N_FEAT = 16
print('U-Net definita.')

In [ ]:
# ── Cell 7: load_static ───────────────────────────────────────────────────
def load_static():
    ds = xr.open_dataset(STATIC_FILE)
    elev_raw = ds['elevation'].values.astype(np.float32)
    slope    = ds['slope'].values.astype(np.float32)
    asp_s    = ds['aspect_sin'].values.astype(np.float32)
    asp_c    = ds['aspect_cos'].values.astype(np.float32)
    ds.close()
    ds_c   = xr.open_dataset(CANOPY_FILE)
    canopy = ds_c['canopy_fraction'].values.astype(np.float32)
    ds_c.close()
    elev_filled = np.where(np.isfinite(elev_raw), elev_raw, np.nanmedian(elev_raw))
    tpi_s = (elev_filled - uniform_filter(elev_filled, size=11, mode='nearest')).astype(np.float32)
    tpi_l = (elev_filled - uniform_filter(elev_filled, size=21, mode='nearest')).astype(np.float32)
    def norm(arr):
        m, s = float(np.nanmean(arr)), float(np.nanstd(arr))
        return ((arr - m) / (s if s > 0 else 1.0)).astype(np.float32)
    elev  = np.nan_to_num(norm(elev_raw), nan=0.0)
    slope = np.nan_to_num(norm(slope),    nan=0.0)
    tpi_s = np.nan_to_num(norm(tpi_s),    nan=0.0)
    tpi_l = np.nan_to_num(norm(tpi_l),    nan=0.0)
    asp_s = np.where(np.isfinite(asp_s), asp_s, 0.0).astype(np.float32)
    asp_c = np.where(np.isfinite(asp_c), asp_c, 0.0).astype(np.float32)
    canopy = np.nan_to_num((canopy - float(np.nanmean(canopy))).astype(np.float32), nan=0.0)
    topo = np.stack([elev, slope, asp_s, asp_c, tpi_s, tpi_l, canopy], axis=0)
    H, W = elev.shape
    lat_n = np.tile(np.linspace(-1, 1, H, dtype=np.float32)[:, None], (1, W))
    lon_n = np.tile(np.linspace(-1, 1, W, dtype=np.float32)[None, :], (H, 1))
    return topo, lat_n, lon_n

print('Carico features statiche...')
topo_static, lat_norm, lon_norm = load_static()
print(f'topo: {topo_static.shape}')

In [ ]:
# ── Cell 8: Helpers ───────────────────────────────────────────────────────
def load_norm_stats(test_year, n_tm=7):
    """Restituisce (mean_arr, std_arr) con n_tm=7 valori.
    Ordine di priorità:
    1) cache .npz salvata da train_loyo40.py
    2) blocco '=== NORM STATS ===' nel log PBS
    3) calcolo diretto dal memmap (fallback lento ma robusto)
    """
    # 1) cache npz
    cache = OUTPUT_DIR / f'norm_stats_cache_test{test_year}.npz'
    if cache.exists():
        d = np.load(str(cache))
        if len(d['mean_arr']) >= n_tm:
            return d['mean_arr'][:n_tm].astype(np.float32), d['std_arr'][:n_tm].astype(np.float32)

    # 2) log PBS — blocco === NORM STATS ===
    fi  = test_year - 1985
    log = LOG_DIR / f'loyo40_job{fi}.log'
    if log.exists():
        text = log.read_text(errors='ignore')
        nb = re.search(r'=== NORM STATS ===(.*?)(?:===|\Z)', text, re.DOTALL)
        if nb:
            pat = re.compile(r'[\w_]+\s*:\s*mean=\s*([\d.]+)\s+std=\s*([\d.]+)')
            rows = pat.findall(nb.group(1))
            if len(rows) >= n_tm:
                mean_arr = np.array([float(r[0]) for r in rows[:n_tm]], dtype=np.float32)
                std_arr  = np.array([float(r[1]) for r in rows[:n_tm]], dtype=np.float32)
                return mean_arr, std_arr
        # fallback: riga singola Stats: (a volte ha tutti i valori)
        m = re.search(r'Stats: mean=\[([^\]]+)\]\s+std=\[([^\]]+)\]', text)
        if m:
            mv = np.array([float(v) for v in m.group(1).split()], dtype=np.float32)
            sv = np.array([float(v) for v in m.group(2).split()], dtype=np.float32)
            if len(mv) >= n_tm:
                return mv[:n_tm], sv[:n_tm]

    # 3) calcola dal memmap dell'anno stesso (ok per visualizzazione)
    print(f'  Nessun cache/log trovato — calcolo stats dal memmap {test_year}...')
    feat = np.load(str(MEMMAP_DIR / f'y{test_year}_feat.npy'), mmap_mode='r')
    mean_arr = np.array([float(np.nanmean(feat[:, c])) for c in range(n_tm)], dtype=np.float32)
    std_arr  = np.array([float(np.nanstd(feat[:, c]))  for c in range(n_tm)], dtype=np.float32)
    std_arr  = np.where(std_arr > 0, std_arr, 1.0)
    return mean_arr, std_arr

def load_model(test_year):
    ckpt  = OUTPUT_DIR / CKPT_TPL.format(year=test_year)
    model = UNet(in_channels=N_FEAT).to(device)
    model.load_state_dict(torch.load(str(ckpt), map_location=device))
    model.eval()
    return model

def load_days(year):
    """Ricostruisce le date da _info.json usando time_start, time_end, T."""
    info_file = MEMMAP_DIR / f'y{year}_info.json'
    with open(info_file) as f:
        info = json.load(f)
    return pd.date_range(start=info['time_start'], end=info['time_end'], periods=info['T'])

print('Helpers definiti.')

In [ ]:
# ── Cell 9: FIG. 8 — Carica memmap per anno e data scelti ────────────────
# Anni consigliati: 1993 (nevoso), 2015 (secco), 1995 (record neve)
MAP_TEST_YEAR  = 1993
MAP_TARGET_DOY = 91    # ~1 aprile

feat_mm = np.load(str(MEMMAP_DIR / f'y{MAP_TEST_YEAR}_feat.npy'), mmap_mode='r')
mask_mm = np.load(str(MEMMAP_DIR / f'y{MAP_TEST_YEAR}_mask.npy'), mmap_mode='r')
days    = load_days(MAP_TEST_YEAR)

doys        = days.day_of_year
t_idx       = int(np.argmin(np.abs(doys - MAP_TARGET_DOY)))
target_date = days[t_idx]
print(f'Anno {MAP_TEST_YEAR}: feat shape={feat_mm.shape}')
print(f'Timestamp scelto: {target_date.date()}  (DOY={doys[t_idx]})')

In [ ]:
# ── Cell 10: Forward pass ─────────────────────────────────────────────────
N_TM = 7
mean_arr, std_arr = load_norm_stats(MAP_TEST_YEAR)
feat_norm = feat_mm[t_idx, :N_TM].copy()
for c in range(N_TM):
    feat_norm[c] = np.nan_to_num(
        (feat_norm[c] - mean_arr[c]) / (std_arr[c] if std_arr[c] > 0 else 1.0),
        nan=0.0, posinf=0.0, neginf=0.0)

inp = np.concatenate([feat_norm, lat_norm[np.newaxis], lon_norm[np.newaxis], topo_static], axis=0)
x_t = torch.from_numpy(inp[np.newaxis]).to(device)

model = load_model(MAP_TEST_YEAR)
with torch.no_grad():
    pred_log = model(x_t).cpu().numpy()[0, 0]
pred_mm_map = np.expm1(pred_log)
obs_raw     = feat_mm[t_idx, 7]
domain_mask = mask_mm[t_idx]

print(f'Pred  mean={pred_mm_map[domain_mask>0].mean():.1f} mm  max={pred_mm_map.max():.1f}')
print(f'Obs   mean={obs_raw[domain_mask>0].mean():.1f} mm  max={obs_raw[domain_mask>0].max():.1f}')

In [ ]:
# ── Cell 11: FIG. 8 — Plot mappa spaziale ────────────────────────────────
vmax      = np.nanpercentile(obs_raw[domain_mask > 0], 98)
pred_plot = np.where(domain_mask > 0, pred_mm_map,           np.nan)
obs_plot  = np.where(domain_mask > 0, obs_raw,               np.nan)
diff_plot = np.where(domain_mask > 0, pred_mm_map - obs_raw, np.nan)
dmax      = np.nanpercentile(np.abs(diff_plot[np.isfinite(diff_plot)]), 95)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
im0 = axes[0].imshow(obs_plot,  origin='upper', cmap='Blues',  vmin=0, vmax=vmax)
im1 = axes[1].imshow(pred_plot, origin='upper', cmap='Blues',  vmin=0, vmax=vmax)
im2 = axes[2].imshow(diff_plot, origin='upper', cmap='RdBu_r', vmin=-dmax, vmax=dmax)

for im, ax, title, label in zip(
        [im0, im1, im2], axes,
        [f'SPORTLIS osservato\n{target_date.date()}',
         f'U-Net predetto (OOS test={MAP_TEST_YEAR})\n{target_date.date()}',
         'Differenza (pred − obs)'],
        ['SWE (mm)', 'SWE (mm)', 'Bias pred−obs (mm)']):
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, label=label, shrink=0.85)

fig.suptitle(
    f'Fig. 8 — Mappa SWE spaziale — {target_date.strftime("%d %b %Y")} — OOS test={MAP_TEST_YEAR}',
    fontsize=13, fontweight='bold')
plt.tight_layout()
out8 = FIG_DIR / f'fig8_map_spatial_{MAP_TEST_YEAR}_{target_date.strftime("%m%d")}.png'
fig.savefig(out8, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvato: {out8}')

In [ ]:
# ── Cell 12: Riepilogo output ─────────────────────────────────────────────
print('=== Figure prodotte ===')
for f in sorted(FIG_DIR.glob('*.png')):
    print(f'  {f.name:55s}  {f.stat().st_size//1024:4d} KB')
print(f'\nCartella: {FIG_DIR}')